In [1]:
#What it does: This cell imports all the necessary Python libraries you need for the project: pandas (for data manipulation), SentenceTransformer (for your BERT model), DBSCAN (for unsupervised clustering), and numpy (for math operations).

#Action: It loads your dataset (synthetic_logs.csv) into a Pandas DataFrame (df) and prints the first 3 rows so you can see what your raw data looks like (timestamp, source, log message, target label).
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import normalize
import numpy as np

# Load dataset
df = pd.read_csv("dataset/synthetic_logs.csv")

df.head(3)


In [2]:
#What it does: It runs df.source.unique() and df.target_label.unique().

#Action: This simply prints out all the unique systems generating the logs (like ModernCRM, AnalyticsEngine) and the unique categories you are trying to predict (like HTTP Status, Critical Error, Security Alert). It helps you understand the scope of the problem.

df.source.unique()
df.target_label.unique()

In [3]:
#What it does: This is where the NLP magic starts. It isolates the log_message column and converts every log into a list of strings.

#Action: It initializes the BERT model (all-MiniLM-L6-v2) and passes all your log messages through it. The model converts the human-readable text into arrays of numbers (embeddings) that capture the context and meaning of the log. Finally, it normalizes these numbers, which is crucial for the clustering algorithm to measure distances accurately.



# Ensure column exists
assert "log_message" in df.columns, "log_message column not found!"

logs = df["log_message"].astype(str).tolist()

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    logs,
    convert_to_numpy=True,
    show_progress_bar=True
)

# Normalize embeddings (CRITICAL)
embeddings = normalize(embeddings)

embeddings[:2]


In [4]:
#What it does: It initializes the DBSCAN machine learning model. This is an "unsupervised" algorithm, meaning you don't give it labels; you just ask it to group similar items together. It groups your log embeddings based on their "cosine" similarity. It assigns a "cluster ID" to each log and adds it as a new column in your DataFrame.


dbscan = DBSCAN(
    eps=0.2,  # start with 0.25–0.35
    min_samples=1,
    metric="cosine"
)

clusters = dbscan.fit_predict(embeddings)

df["cluster"] = clusters
df.head(3)


In [5]:
#What it does: You filter the DataFrame to only show rows that were assigned to "Cluster 5". This is just a visual check to see if the algorithm successfully grouped similar logs together (in this case, it grouped a bunch of "Resource Usage" OpenStack logs).

df[df.cluster == 5]

In [6]:
#What it does: It counts how many logs are in each cluster and filters out the small ones, only keeping "large clusters" (clusters with more than 10 logs).

#Action: It loops through these large clusters and prints the first 5 log messages from each.
#By looking at these large, repetitive clusters, you can easily spot the fixed patterns, which tells you exactly what Regular Expressions (Regex) you need to write in the next step.



cluster_counts = df["cluster"].value_counts()

# Get cluster IDs with more than 10 logs
large_clusters = cluster_counts[cluster_counts > 10].index

for cluster in large_clusters:
    print(f"===== Cluster {cluster} | Total logs: {cluster_counts[cluster]} =====")

    sample_logs = df[df["cluster"] == cluster]["log_message"].head(5)

    print(sample_logs.to_string(index=False))
    print()


In [7]:
import re


def classify_with_regex(log_message: str):
    regex_patterns = {
        # ---------- OpenStack ----------
        r"^nova\.osapi_compute\.wsgi\.server.*": "OpenStack API Server",
        r"^nova\.compute\.claims.*": "OpenStack Compute Claims",
        r"^nova\.compute\.resource_tracker.*": "OpenStack Resource Tracker",

        # ---------- USER ACTIONS ----------
        r"User\s+User\d+\s+logged\s+(in|out)\.?": "User Action",
        r"Account with ID\s+\w+\s+created by\s+.*": "User Action",

        # ---------- AUTH / SECURITY ----------
        r"(Multiple|Unauthorized|Invalid|Denied).*login.*": "Authentication Failure",
        r"(has escalated|privilege).*admin": "Privilege Escalation",

        # ---------- SYSTEM NOTIFICATIONS ----------
        r"Backup\s+(started|ended)\s+at\s+.*": "System Notification",
        r"Backup completed successfully\.": "System Notification",
        r"System updated to version\s+.*": "System Notification",
        r"Disk cleanup completed successfully\.": "System Notification",
        r"System reboot initiated by user\s+.*": "System Notification",
        r"File\s+.*\s+uploaded successfully by user\s+.*": "System Notification",

        # ---------- FAILURES ----------
        r"(Replication|replication|Shard).*failed|did not complete": "Replication Failure",
        r"(Critical|Essential|Failure).*system.*component": "Critical System Failure",
        r"(RAID|disk).*failure|faulty disks": "Disk / RAID Failure",
        r"(kernel panic|boot process|boot sequence).*failed": "Kernel / Boot Failure",
        r"System configuration.*(invalid|corrupted|failure|errors)": "Configuration Error",

        # ---------- SERVICE ISSUES ----------
        r"(Email|Mail).*sending.*(issue|fault|error|glitch)": "Email Service Issue",
        r"(SSL|health check).*failed": "Service Health / SSL Issue",
        r"Module\s+X.*(invalid|mismatch|format)": "Module Input Error",

        # ---------- SECURITY ALERTS ----------
        r"(Abnormal|Anomalous|suspicious|Security alert).*server": "Security Alert"
    }

    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message, re.IGNORECASE):
            return label

    return "Unknown"


In [8]:
#Tests the function on a single string to make sure it works (it correctly outputs 'User Action').

classify_with_regex("Account with ID 5351 created by User63")


In [9]:
#It applies this Regex function to your entire dataset and saves the predictions in a new column called regex_label. This is Tier 1 of your hybrid system!

df["regex_label"] = df["log_message"].apply(classify_with_regex)

df

In [10]:
#What it does: Not every log can be solved by Regex. This cell creates a new DataFrame (df_non_regex) that only contains the logs that the Regex function failed to classify (the ones labeled "Unknown").

#Action: It prints the shape. This means out of your 2,410 total logs, 746 slipped past the Regex and need to be handled by BERT or the LLM.

df_non_regex = df[df["regex_label"] == "Unknown"].copy()

df_non_regex.shape



In [11]:
#What it does: Not all logs are easy to train a model on. Some systems like 'LegacyCRM' have very few examples. This cell separates those rare logs from the ones we will use to train our AI model.

#Action: It creates two subsets: one for LegacyCRM (LLM fallback) and one for everything else (BERT training). It then prints the count of logs in each group.

# Separate LegacyCRM (LLM fallback) from other sources (BERT training)
df_legacy = df_non_regex[df_non_regex['source'] == 'LegacyCRM'].copy()
df_bert = df_non_regex[df_non_regex['source'] != 'LegacyCRM'].copy()

print(f"Logs for LLM Fallback (LegacyCRM): {df_legacy.shape[0]}")
print(f"Logs for BERT Training (Other sources): {df_bert.shape[0]}")

In [12]:
#What it does: This is where we train our Tier 2 Machine Learning model. It uses the BERT embeddings we generated earlier to teach a Logistic Regression model how to recognize different log categories.

#Action: It encodes the logs, splits them into training (80%) and testing (20%) sets, trains the model, and then prints a 'Classification Report' showing how accurate the model is.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Generate BERT embeddings for the logs we'll use for training
X = model.encode(df_bert['log_message'].tolist(), show_progress_bar=True)
y = df_bert['target_label']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

# Evaluate performance
y_pred = lr_model.predict(X_test)
print(classification_report(y_test, y_pred))

In [13]:
#What it does: Once the model is trained, we need to save it so our web server (FastAPI) can use it later without needing to retrain it every time.

#Action: It creates a 'models' folder and saves the trained model as a '.joblib' file.
import joblib
import os

# Create models directory if it doesn't exist
os.makedirs("../models", exist_ok=True)

# Save the model to disk
joblib.dump(lr_model, "../models/log_classifier.joblib")
print("Model successfully saved to models/log_classifier.joblib")